[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/ai-forever/FRIDA-Decisions/blob/grok/examples-rag_filtering/examples/rag_filtering/rag_filtering.ipynb)

# Filter passages before they reach the answering model

A retriever returns a shortlist. Something has to throw away the passages that do not answer, including the case where none of them do. An LLM would read the whole shortlist and quote whatever looks close. FRIDA-Decisions ranks the passages in one request, and a yes/no question on each passage gives an absolute probability.

`ranking` probabilities split one unit across the list, so they cannot filter: a confident winner in a long list still looks small. `scores` are raw margins. Subtract the mean of the list, as in the repository README, and remember that this constant belongs to the shape of the list. It still cannot say that nothing is relevant. That is what `noul` is for.

Patterns: [Classifying RAG passages](https://docs.typesafe.ai/cookbooks/classifying_rag_passages) and [Re-ranking](https://docs.typesafe.ai/cookbooks/rerank_typesafe).


In [1]:
import sys
!{sys.executable} -m pip install -q "frida-decisions[torch] @ git+https://github.com/ai-forever/FRIDA-Decisions@v0.4.0" rank_bm25


In [2]:
import json, os
from pathlib import Path
import torch
import frida_decisions
from frida_decisions import Judge

MODEL = os.environ.get("FD_MODEL", "ai-forever/FRIDA-Decisions")   # repo id or a local folder
device = "cuda" if torch.cuda.is_available() else "cpu"
if device == "cpu":
    torch.set_num_threads(min(4, os.cpu_count() or 1))
judge = Judge.from_pretrained(MODEL, device=device)
gpu = f" ({torch.cuda.get_device_name(0)})" if device == "cuda" else ""
print(f"frida-decisions {frida_decisions.__version__}, loaded on {device}{gpu}, {judge.dtype}")


Reconstructing (incomplete total...): |          |  0.00B /  0.00B            

Fetching 6 files:   0%|          | 0/6 [00:00<?, ?it/s]

Loading weights:   0%|          | 0/219 [00:00<?, ?it/s]

frida-decisions 0.4.0, loaded on cpu, torch.float32


In [3]:
try:
    from rank_bm25 import BM25Okapi
except ImportError as error:
    raise SystemExit("pip install rank_bm25") from error

TOP_K = 6
# Fitted in the README on lists of five with one relevant passage. Re-fit it for
# a list of a different shape; it is not a universal cutoff.
LIST_THRESHOLD = 3.5
NOUL_THRESHOLD = 0.5


def tokens(text: str) -> list[str]:
    return text.lower().replace("ё", "е").split()


def shortlist(question: str, passages: list[dict], k: int) -> list[dict]:
    index = BM25Okapi([tokens(item["text"]) for item in passages])
    scores = index.get_scores(tokens(question))
    order = sorted(range(len(passages)), key=lambda i: -scores[i])[:k]
    return [passages[i] for i in order]


def request_for(question: str, picked: list[dict]) -> dict:
    # The ranking id is not shown to the model. The noul instruction repeats a
    # short passage, so the passages in data.json are one sentence each.
    questions = {
        "best": {
            "type": "ranking",
            "instructions": "Какой фрагмент лучше всего отвечает на вопрос?",
            "criteria": {item["id"]: item["text"] for item in picked},
        },
    }
    for item in picked:
        questions[item["id"]] = {
            "type": "noul",
            "instructions": "Этот фрагмент отвечает на вопрос? " + item["text"],
        }
    return {"state": question, "questions": questions}


def hit(order: list[str], relevant: set[str], k: int):
    if not relevant:
        return None
    return int(any(item in relevant for item in order[:k]))


## Relative rank, then an absolute yes/no

Top-k hit rate compares BM25 with `ranking` on queries that have a relevant passage.
The list threshold only looks at margins inside one shortlist. The noul gate is what
drops a query whose shortlist does not answer it.


In [4]:
def run(judge, rows: dict) -> None:
    passages = rows["passages"]
    by_id = {item["id"]: item["text"] for item in passages}
    top1 = {"bm25": [], "ranking": []}
    top3 = {"bm25": [], "ranking": []}
    kept_relevant = kept_total = gold_relevant = 0
    empty_queries = empty_abstained = 0

    for query in rows["queries"]:
        relevant = set(query["relevant"])
        picked = shortlist(query["question"], passages, TOP_K)
        bm25_order = [item["id"] for item in picked]
        response = judge(request_for(query["question"], picked))
        answer = response["answers"]["best"]
        scores = answer["scores"]
        level = sum(scores.values()) / len(scores)
        ranking_order = answer["ranking"]
        print(f"\n{query['question']}")
        print(f"  bm25    {bm25_order}")
        for key in ranking_order:
            distance = scores[key] - level
            noul = response["answers"][key]["noul"]
            flags = []
            if distance >= LIST_THRESHOLD:
                flags.append("above-list")
            if noul >= NOUL_THRESHOLD:
                flags.append("noul")
            if key in relevant:
                flags.append("gold")
            print(f"  {key:4} margin {scores[key]:+6.2f}  above list {distance:+5.2f}  "
                  f"noul {noul:.2f}  {' '.join(flags)}")

        if relevant:
            top1["bm25"].append(hit(bm25_order, relevant, 1))
            top1["ranking"].append(hit(ranking_order, relevant, 1))
            top3["bm25"].append(hit(bm25_order, relevant, 3))
            top3["ranking"].append(hit(ranking_order, relevant, 3))
            for key in ranking_order:
                if scores[key] - level < LIST_THRESHOLD:
                    continue
                kept_total += 1
                kept_relevant += key in relevant
            gold_relevant += len(relevant & set(bm25_order))
        else:
            empty_queries += 1
            if all(response["answers"][key]["noul"] < NOUL_THRESHOLD for key in ranking_order):
                empty_abstained += 1

        if response["usage"]["state_truncated"]:
            print("  state truncated")

    def mean(values: list) -> str:
        return "n/a" if not values else f"{sum(values) / len(values):.0%}"

    print("\nqueries that have a relevant passage:")
    print(f"  top-1  bm25 {mean(top1['bm25'])}  ranking {mean(top1['ranking'])}")
    print(f"  top-3  bm25 {mean(top3['bm25'])}  ranking {mean(top3['ranking'])}")
    precision = "n/a" if not kept_total else f"{kept_relevant / kept_total:.0%}"
    recall = "n/a" if not gold_relevant else f"{kept_relevant / gold_relevant:.0%}"
    print(f"  list threshold {LIST_THRESHOLD}: kept {kept_total}, "
          f"precision {precision}, recall against the shortlist {recall}")
    print(f"queries with nothing relevant: abstained on noul "
          f"{empty_abstained}/{empty_queries} (cutoff {NOUL_THRESHOLD})")


In [5]:
def load_rows():
    import urllib.request
    path = Path("data.json")
    if not path.exists():
        url = ("https://raw.githubusercontent.com/ai-forever/FRIDA-Decisions/"
               "grok/examples-rag-filtering/examples/rag_filtering/data.json")
        path.write_bytes(urllib.request.urlopen(url, timeout=60).read())
    return json.loads(path.read_text(encoding="utf-8"))

run(judge, load_rows())



Как удалить накипь из чайника?
  bm25    ['k3', 'k1', 'k2', 'k4', 'k5', 'k6']
  k2   margin  +6.79  above list +7.21  noul 0.90  above-list noul gold
  k1   margin  +4.41  above list +4.83  noul 0.90  above-list noul gold
  k3   margin  +0.78  above list +1.20  noul 0.35  
  k5   margin  -4.42  above list -4.00  noul 0.01  
  k4   margin  -4.84  above list -4.42  noul 0.01  
  k6   margin  -5.25  above list -4.83  noul 0.04  

Как вернуть деньги за заказ?
  bm25    ['k5', 'k4', 'k9', 'k1', 'k2', 'k3']
  k5   margin  +4.31  above list +4.85  noul 0.92  above-list noul gold
  k4   margin  +2.88  above list +3.42  noul 0.77  noul gold
  k9   margin  -0.91  above list -0.38  noul 0.11  
  k2   margin  -2.28  above list -1.74  noul 0.13  
  k1   margin  -2.41  above list -1.88  noul 0.09  
  k3   margin  -4.81  above list -4.28  noul 0.03  

Не приходит письмо для сброса пароля.
  bm25    ['k7', 'k12', 'k6', 'k1', 'k2', 'k3']
  k7   margin  +3.34  above list +5.74  noul 0.26  above-list go